# RSM-8443 Hot Delivery
**Optimizing Supply Chain Management and Logistics — Winter 2026**

In [1]:
!pip install pulp -q

In [2]:
import pulp
import pandas as pd
from datetime import datetime
import time as timer
import os

# ── Global constants ──────────────────────────────────────────────────────────
# All CSV files live in the same folder as this notebook.
BASE = os.path.dirname(os.path.abspath(__file__)) + "/" if "__file__" in dir() else ""

SERVICE = 5  # minutes per customer stop (Parts II-IV)

def parse_time(s):
    """'2022-04-02 7:27 PM'  →  minutes from midnight"""
    dt = datetime.strptime(s.strip(), "%Y-%m-%d %I:%M %p")
    return dt.hour * 60 + dt.minute

def load_distances():
    df = pd.read_csv(BASE + "distances.csv")
    d = {}
    for _, row in df.iterrows():
        d[(row["origin"], row["destination"])] = row["distance"]
        d[(row["destination"], row["origin"])]  = row["distance"]
    return d

def dist_km(d_map, i, j, depot_internal=None, depot_loc=None):
    """Distance lookup. Maps internal depot tag → actual location name."""
    if depot_internal:
        i = depot_loc if i == depot_internal else i
        j = depot_loc if j == depot_internal else j
    if i == j: return 0.0
    return d_map.get((i, j), 1e9)

dist_map = load_distances()
print(f"Distance matrix loaded: {len(dist_map)//2} neighborhood pairs")


Distance matrix loaded: 5151 neighborhood pairs


---
## Part I — Single-Driver Perspective

**Model:** Open-path TSP with pickup-before-delivery constraints.  
- Driver starts at **Downtown Toronto (Rosedale)**, ends at last customer location.  
- Dummy `SINK` node (zero-cost arcs from customers) handles the open path.  
- **MTZ constraints** eliminate subtours.  
- **Precedence:** `u[restaurant_k] + 1 ≤ u[customer_k]` for each order.  
- **Objective:** minimize total distance.

In [3]:
def solve_part1(orders_file):
    DEPOT = "__DEPOT__"
    DEPOT_LOC = "Downtown Toronto (Rosedale)"
    SINK  = "__SINK__"

    orders_df = pd.read_csv(BASE + orders_file)
    orders = list(zip(orders_df["restaurant"], orders_df["customer"]))
    restaurants = [r for r, c in orders]
    customers   = [c for r, c in orders]
    customer_set = set(customers)

    seen, locs = set(), []
    for r, c in orders:
        for loc in [r, c]:
            if loc not in seen:
                seen.add(loc); locs.append(loc)

    all_nodes = [DEPOT] + locs + [SINK]
    real = locs; n = len(real); M = n + 2

    def d(i, j):
        return dist_km(dist_map, i, j, depot_internal=DEPOT, depot_loc=DEPOT_LOC)

    arcs = [
        (i, j) for i in all_nodes for j in all_nodes
        if i != j and j != DEPOT and i != SINK
        and not (j == SINK and i not in customer_set)
    ]

    prob = pulp.LpProblem("Part1", pulp.LpMinimize)
    x = {(i,j): pulp.LpVariable(f"x_{i}_{j}", cat="Binary") for (i,j) in arcs}
    u = {v: pulp.LpVariable(f"u_{v}", lowBound=1, upBound=n) for v in real}

    prob += pulp.lpSum((0.0 if j==SINK else d(i,j)) * x[i,j] for (i,j) in arcs)
    prob += pulp.lpSum(x[i,j] for (i,j) in arcs if i==DEPOT) == 1
    prob += pulp.lpSum(x[i,j] for (i,j) in arcs if j==SINK)  == 1
    for v in real:
        prob += pulp.lpSum(x[i,j] for (i,j) in arcs if j==v) == 1
        prob += pulp.lpSum(x[i,j] for (i,j) in arcs if i==v) == 1
    for (i,j) in arcs:
        if i in real and j in real:
            prob += u[j] >= u[i] + 1 - M*(1 - x[i,j])
    for r, c in orders:
        prob += u[c] >= u[r] + 1

    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=120))

    print(f"\n{'='*60}\nInstance : {orders_file}")
    print(f"Status   : {pulp.LpStatus[prob.status]}")
    print(f"Distance : {pulp.value(prob.objective):.2f} km")

    route, cur = [DEPOT_LOC], DEPOT
    while True:
        nxt = next((j for (i,j) in arcs if i==cur and pulp.value(x[i,j])>0.5), None)
        if nxt is None or nxt == SINK: break
        route.append(nxt); cur = nxt

    roles = {}
    for k, (r, c) in enumerate(orders):
        roles.setdefault(r, []).append(f"PICKUP  order {k+1}")
        roles.setdefault(c, []).append(f"DELIVER order {k+1}")

    print("\nRoute:")
    for step, node in enumerate(route):
        tag = ", ".join(roles.get(node, ["START"]))
        print(f"  {step+1:2d}. [{tag}]  {node}")

    return pulp.value(prob.objective)

# ── Run Part I ────────────────────────────────────────────────────────────────
solve_part1("part1_ordersA.csv")
solve_part1("part1_ordersB.csv")



Instance : part1_ordersA.csv
Status   : Optimal
Distance : 4.65 km

Route:
   1. [START]  Downtown Toronto (Rosedale)
   2. [PICKUP  order 1]  Downtown Toronto (Underground city)
   3. [DELIVER order 1]  Downtown Toronto (Central Bay Street)

Instance : part1_ordersB.csv
Status   : Optimal
Distance : 33.88 km

Route:
   1. [START]  Downtown Toronto (Rosedale)
   2. [PICKUP  order 1, PICKUP  order 4]  Downtown Toronto (Central Bay Street)
   3. [DELIVER order 4, PICKUP  order 5]  Downtown Toronto (Richmond / Adelaide / King)
   4. [DELIVER order 1]  Downtown Toronto (Underground city)
   5. [DELIVER order 5]  Downtown Toronto (St. James Town / Cabbagetown)
   6. [PICKUP  order 3]  York (Cedarvale)
   7. [DELIVER order 3]  Central Toronto (The Annex / North Midtown / Yorkville)
   8. [PICKUP  order 2]  Etobicoke Northwest (Clairville / Humberwood / Woodbine Downs / West Humber / Kipling Heights / Rexdale / Elms / Tandridge / Old Rexdale)
   9. [DELIVER order 2]  Etobicoke (South Steeles

33.875942274546034

---
## Part II — A Matter of Time

**New elements vs Part I:**
- `t[v]`: arrival time at each node (minutes from midnight)
- Food availability constraint: `t[restaurant_k] ≥ avail_k`
- Customer wait time: `t[customer_k] − avail_k`
- Service time: **5 min** per customer stop
- Speed: **40 km/h**
- Constraint: **average wait time ≤ W**

**Questions addressed:**
1. How does difficulty scale with W?
2. Trade-off curve: W vs distance
3. Problem with average wait metric + alternative (max wait)

In [4]:
DEPOT2     = "__DEPOT__"
DEPOT2_LOC = "Downtown Toronto (Rosedale)"
SINK2      = "__SINK__"
SPEED2     = 40   # km/h

def solve_part2(orders_file, W=120, use_max_wait=False, time_limit=120, verbose=False):
    orders_df = pd.read_csv(BASE + orders_file)
    orders = list(zip(
        orders_df["restaurant"],
        orders_df["customer"],
        [parse_time(a) for a in orders_df["estimated availability"]]
    ))
    n_orders = len(orders)
    customers = [c for r,c,a in orders]
    customer_set = set(customers)

    seen, locs = set(), []
    for r, c, a in orders:
        for loc in [r, c]:
            if loc not in seen: seen.add(loc); locs.append(loc)

    all_nodes = [DEPOT2] + locs + [SINK2]
    real = locs; n = len(real)
    M_pos = n + 2; M_time = 24*60

    def d2(i, j):
        return dist_km(dist_map, i, j, depot_internal=DEPOT2, depot_loc=DEPOT2_LOC)
    def tt2(i, j):
        return d2(i, j) / SPEED2 * 60.0

    arcs = [(i,j) for i in all_nodes for j in all_nodes
            if i!=j and j!=DEPOT2 and i!=SINK2
            and not (j==SINK2 and i not in customer_set)]

    prob = pulp.LpProblem("Part2", pulp.LpMinimize)
    x = {(i,j): pulp.LpVariable(f"x_{i}_{j}", cat="Binary") for (i,j) in arcs}
    u = {v: pulp.LpVariable(f"u_{v}", lowBound=1, upBound=n) for v in real}
    t = {v: pulp.LpVariable(f"t_{v}", lowBound=0, upBound=M_time)
         for v in all_nodes if v != SINK2}

    prob += pulp.lpSum((0.0 if j==SINK2 else d2(i,j))*x[i,j] for (i,j) in arcs)
    prob += pulp.lpSum(x[i,j] for (i,j) in arcs if i==DEPOT2) == 1
    prob += pulp.lpSum(x[i,j] for (i,j) in arcs if j==SINK2)  == 1
    for v in real:
        prob += pulp.lpSum(x[i,j] for (i,j) in arcs if j==v) == 1
        prob += pulp.lpSum(x[i,j] for (i,j) in arcs if i==v) == 1
    for (i,j) in arcs:
        if i in real and j in real:
            prob += u[j] >= u[i] + 1 - M_pos*(1 - x[i,j])
    for r, c, a in orders:
        prob += u[c] >= u[r] + 1

    svc = {v: (SERVICE if v in customer_set else 0) for v in all_nodes if v!=SINK2}
    for (i,j) in arcs:
        if j!=SINK2 and i in t and j in t:
            prob += t[j] >= t[i] + svc.get(i,0) + tt2(i,j) - M_time*(1-x[i,j])
    for r, c, a in orders:
        if r in t: prob += t[r] >= a

    if use_max_wait:
        for r, c, a in orders: prob += t[c] - a <= W
    else:
        prob += pulp.lpSum(t[c]-a for r,c,a in orders) <= n_orders * W

    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=time_limit))
    if prob.status != 1:
        return None, None, None, pulp.LpStatus[prob.status]

    total_dist = pulp.value(prob.objective)
    wait_times = [pulp.value(t[c])-a for r,c,a in orders]
    avg_wait, max_wait = sum(wait_times)/n_orders, max(wait_times)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Instance : {orders_file}  |  W={W}  |  {'max' if use_max_wait else 'avg'} wait")
        print(f"Status   : {pulp.LpStatus[prob.status]}")
        print(f"Distance : {total_dist:.2f} km   Avg wait: {avg_wait:.1f} min   Max wait: {max_wait:.1f} min")
        route, cur = [DEPOT2_LOC], DEPOT2
        while True:
            nxt = next((j for (i,j) in arcs if i==cur and pulp.value(x.get((i,j),0))>0.5), None)
            if nxt is None or nxt==SINK2: break
            route.append(nxt); cur = nxt
        roles = {}
        for k,(r,c,a) in enumerate(orders):
            roles.setdefault(r,[]).append(f"PICKUP  order {k+1} (ready {a}min)")
            roles.setdefault(c,[]).append(f"DELIVER order {k+1} (wait {wait_times[k]:.1f}min)")
        print("\nRoute:")
        for step, node in enumerate(route):
            tag = ", ".join(roles.get(node, ["START"]))
            arr = f"arrive {pulp.value(t[node]):.0f}min" if node in t else ""
            print(f"  {step+1:2d}. [{tag}]  {node}  [{arr}]")
    return total_dist, avg_wait, max_wait, pulp.LpStatus[prob.status]

def tradeoff_curve(orders_file, W_values=None):
    if W_values is None: W_values = list(range(20, 301, 20))
    print(f"\nTrade-off curve: {orders_file}")
    print(f"{'W':>8}  {'Distance':>12}  {'Avg wait':>10}  {'Status':>10}")
    results = []
    for W in W_values:
        d, avg, mx, status = solve_part2(orders_file, W=W)
        if d is not None:
            print(f"{W:>8}  {d:>12.2f}  {avg:>10.1f}  {status:>10}")
            results.append((W, d, avg))
        else:
            print(f"{W:>8}  {'infeasible':>12}  {'—':>10}  {status:>10}")
    return results

def compare_metrics(orders_file, W=120):
    print(f"\n{'='*60}\nMetric comparison: {orders_file}  (W={W})")
    d_avg, a_avg, m_avg, _ = solve_part2(orders_file, W=W, use_max_wait=False, verbose=True)
    d_max, a_max, m_max, _ = solve_part2(orders_file, W=W, use_max_wait=True,  verbose=True)
    print(f"\n  Avg-wait model : dist={d_avg:.2f} km, avg={a_avg:.1f} min, max={m_avg:.1f} min")
    print(f"  Max-wait model : dist={d_max:.2f} km, avg={a_max:.1f} min, max={m_max:.1f} min")

# ── Run Part II ───────────────────────────────────────────────────────────────
solve_part2("part2_ordersA.csv", W=120, verbose=True)
solve_part2("part2_ordersB.csv", W=120, verbose=True)



Instance : part2_ordersA.csv  |  W=120  |  avg wait
Status   : Optimal
Distance : 42.77 km   Avg wait: 120.0 min   Max wait: 224.5 min

Route:
   1. [START]  Downtown Toronto (Rosedale)  []
   2. [PICKUP  order 2 (ready 1230min)]  Central Toronto (North Toronto West)  [arrive 1230min]
   3. [DELIVER order 2 (wait 15.5min)]  Etobicoke (Westmount)  [arrive 1246min]
   4. [PICKUP  order 1 (ready 1167min)]  Scarborough (Kennedy Park / Ionview / East Birchmount Park)  [arrive 1283min]
   5. [DELIVER order 1 (wait 224.5min)]  Scarborough (Woburn)  [arrive 1391min]



Instance : part2_ordersB.csv  |  W=120  |  avg wait
Status   : Optimal
Distance : 51.40 km   Avg wait: 78.6 min   Max wait: 137.4 min

Route:
   1. [DELIVER order 3 (wait 11.8min)]  Downtown Toronto (Rosedale)  [arrive 1109min]
   2. [PICKUP  order 2 (ready 1069min)]  North York (Sweeney Park / Wigmore Park)  [arrive 1069min]
   3. [PICKUP  order 3 (ready 1097min)]  Scarborough (The Golden Mile / Clairlea / Oakridge / Birchmount Park East)  [arrive 1097min]
   4. [DELIVER order 3 (wait 11.8min)]  Downtown Toronto (Rosedale)  [arrive 1109min]
   5. [PICKUP  order 4 (ready 1073min)]  Etobicoke (Westmount)  [arrive 1133min]
   6. [PICKUP  order 1 (ready 1037min)]  Etobicoke (Islington Avenue)  [arrive 1138min]
   7. [DELIVER order 4 (wait 68.8min)]  Etobicoke (West Deane Park / Princess Gardens / Martin Grove / Islington / Cloverdale)  [arrive 1142min]
   8. [DELIVER order 2 (wait 96.3min)]  Downtown Toronto (University of Toronto / Harbord)  [arrive 1165min]
   9. [DELIVER order 1 (wait

(51.40200682468172, 78.56004999999999, 137.35500000000002, 'Optimal')

In [5]:
# Q2: Trade-off curves (W vs distance)
tradeoff_curve("part2_ordersA.csv", W_values=list(range(20, 301, 20)))
tradeoff_curve("part2_ordersB.csv", W_values=list(range(20, 301, 20)))



Trade-off curve: part2_ordersA.csv
       W      Distance    Avg wait      Status
      20         43.32        20.0     Optimal
      40         43.32        40.0     Optimal
      60         43.32        60.0     Optimal
      80         42.77        80.0     Optimal
     100         42.77       100.0     Optimal
     120         42.77       120.0     Optimal
     140         42.77       140.0     Optimal
     160         42.77       144.3     Optimal
     180         42.77       144.3     Optimal
     200         42.77       144.3     Optimal
     220         42.77       144.3     Optimal
     240         42.77       144.3     Optimal
     260         42.77       144.3     Optimal
     280         42.77       144.3     Optimal
     300         42.77       144.3     Optimal

Trade-off curve: part2_ordersB.csv
       W      Distance    Avg wait      Status


      20    infeasible           —  Infeasible


      40         65.91        40.0     Optimal


      60         55.63        58.4     Optimal


      80         51.40        78.6     Optimal


...

     240         51.40        78.6     Optimal


     260         51.40        78.6     Optimal


     280         51.40        78.6     Optimal


     300         51.40       145.0     Optimal


[(40, 65.90619703138421, 40.0),
 (60, 55.628504305056005, 58.39594999999997),
 (80, 51.40200682468172, 78.56004999999999),
 (100, 51.40200682468172, 78.56004999999999),
 (120, 51.40200682468172, 78.56004999999999),
 (140, 51.40200682468172, 78.56004999999999),
 (160, 51.40200682468172, 78.56004999999999),
 (180, 51.40200682468172, 78.56004999999999),
 (200, 51.40200682468172, 78.56004999999999),
 (220, 51.40200682468172, 78.56004999999999),
 (240, 51.40200682468172, 78.56004999999999),
 (260, 51.40200682468172, 78.56004999999999),
 (280, 51.40200682468172, 78.56004999999999),
 (300, 51.40200682468172, 144.97129999999999)]

In [6]:
# Q3: Alternative metric — max wait instead of average wait
compare_metrics("part2_ordersA.csv", W=120)
compare_metrics("part2_ordersB.csv", W=120)



Metric comparison: part2_ordersA.csv  (W=120)

Instance : part2_ordersA.csv  |  W=120  |  avg wait
Status   : Optimal
Distance : 42.77 km   Avg wait: 120.0 min   Max wait: 224.5 min

Route:
   1. [START]  Downtown Toronto (Rosedale)  []
   2. [PICKUP  order 2 (ready 1230min)]  Central Toronto (North Toronto West)  [arrive 1230min]
   3. [DELIVER order 2 (wait 15.5min)]  Etobicoke (Westmount)  [arrive 1246min]
   4. [PICKUP  order 1 (ready 1167min)]  Scarborough (Kennedy Park / Ionview / East Birchmount Park)  [arrive 1283min]
   5. [DELIVER order 1 (wait 224.5min)]  Scarborough (Woburn)  [arrive 1391min]

Instance : part2_ordersA.csv  |  W=120  |  max wait
Status   : Optimal
Distance : 43.32 km   Avg wait: 64.6 min   Max wait: 120.0 min

Route:
   1. [START]  Downtown Toronto (Rosedale)  []
   2. [PICKUP  order 1 (ready 1167min)]  Scarborough (Kennedy Park / Ionview / East Birchmount Park)  [arrive 1167min]
   3. [DELIVER order 1 (wait 9.1min)]  Scarborough (Woburn)  [arrive 1176min]



Instance : part2_ordersB.csv  |  W=120  |  avg wait
Status   : Optimal
Distance : 51.40 km   Avg wait: 78.6 min   Max wait: 137.4 min

Route:
   1. [DELIVER order 3 (wait 11.8min)]  Downtown Toronto (Rosedale)  [arrive 1109min]
   2. [PICKUP  order 2 (ready 1069min)]  North York (Sweeney Park / Wigmore Park)  [arrive 1069min]
   3. [PICKUP  order 3 (ready 1097min)]  Scarborough (The Golden Mile / Clairlea / Oakridge / Birchmount Park East)  [arrive 1097min]
   4. [DELIVER order 3 (wait 11.8min)]  Downtown Toronto (Rosedale)  [arrive 1109min]
   5. [PICKUP  order 4 (ready 1073min)]  Etobicoke (Westmount)  [arrive 1133min]
   6. [PICKUP  order 1 (ready 1037min)]  Etobicoke (Islington Avenue)  [arrive 1138min]
   7. [DELIVER order 4 (wait 68.8min)]  Etobicoke (West Deane Park / Princess Gardens / Martin Grove / Islington / Cloverdale)  [arrive 1142min]
   8. [DELIVER order 2 (wait 96.3min)]  Downtown Toronto (University of Toronto / Harbord)  [arrive 1165min]
   9. [DELIVER order 1 (wait


Instance : part2_ordersB.csv  |  W=120  |  max wait
Status   : Optimal
Distance : 55.63 km   Avg wait: 62.1 min   Max wait: 120.0 min

Route:
   1. [DELIVER order 3 (wait 41.5min)]  Downtown Toronto (Rosedale)  [arrive 1138min]
   2. [PICKUP  order 4 (ready 1073min)]  Etobicoke (Westmount)  [arrive 1073min]
   3. [DELIVER order 4 (wait 7.8min)]  Etobicoke (West Deane Park / Princess Gardens / Martin Grove / Islington / Cloverdale)  [arrive 1081min]
   4. [PICKUP  order 1 (ready 1037min)]  Etobicoke (Islington Avenue)  [arrive 1090min]
   5. [PICKUP  order 2 (ready 1069min)]  North York (Sweeney Park / Wigmore Park)  [arrive 1117min]
   6. [PICKUP  order 3 (ready 1097min)]  Scarborough (The Golden Mile / Clairlea / Oakridge / Birchmount Park East)  [arrive 1127min]
   7. [DELIVER order 3 (wait 41.5min)]  Downtown Toronto (Rosedale)  [arrive 1138min]
   8. [DELIVER order 2 (wait 79.0min)]  Downtown Toronto (University of Toronto / Harbord)  [arrive 1148min]
   9. [DELIVER order 1 (wait 

---
## Part III — The More the Merrier

**Extension of Part II to multiple drivers:**
- Each driver has their own starting location and velocity.
- Model simultaneously **assigns** orders to drivers and **routes** each driver.
- Uses **order-indexed nodes** (R_k, C_k) to avoid location collisions.
- Wait time linearization: `w[d,k] = y[d,k] × (t[d,C_k] − avail_k)`
- Constraint: **global average wait time ≤ W**
- **Objective:** minimize total distance across all drivers.

In [7]:
def solve_part3(orders_file, drivers_file, W=120, time_limit=120, verbose=False, n_drivers=None):
    orders_df  = pd.read_csv(BASE + orders_file)
    drivers_df = pd.read_csv(BASE + drivers_file)
    if n_drivers is not None: drivers_df = drivers_df.iloc[:n_drivers]

    orders = list(zip(
        orders_df["restaurant"], orders_df["customer"],
        [parse_time(a) for a in orders_df["estimated availability"]]
    ))
    K = len(orders); D = len(drivers_df)
    d_locs = list(drivers_df["start region"]); d_speeds = list(drivers_df["velocity"])

    def dep(d): return f"__DEP_{d}__"
    def snk(d): return f"__SNK_{d}__"
    def R(k):   return f"R{k}"
    def C(k):   return f"C{k}"

    rest_nodes = [R(k) for k in range(K)]; cust_nodes = [C(k) for k in range(K)]
    real_nodes = rest_nodes + cust_nodes; cust_set = set(cust_nodes)

    def node_loc(drv, v):
        if v == dep(drv): return d_locs[drv]
        k = int(v[1:])
        return orders[k][0] if v.startswith("R") else orders[k][1]

    def arc_dist(drv, i, j):
        if j == snk(drv): return 0.0
        return dist_km(dist_map, node_loc(drv,i), node_loc(drv,j))

    def arc_tt(drv, i, j):
        return arc_dist(drv,i,j) / d_speeds[drv] * 60.0

    def build_arcs(drv):
        nodes = [dep(drv)] + real_nodes + [snk(drv)]
        return [(i,j) for i in nodes for j in nodes
                if i!=j and j!=dep(drv) and i!=snk(drv)
                and not (j==snk(drv) and i not in cust_set)]

    arcs_d = {drv: build_arcs(drv) for drv in range(D)}

    prob = pulp.LpProblem("Part3", pulp.LpMinimize)
    M_pos = 2*K+2; M_time = 24*60

    y = {(drv,k): pulp.LpVariable(f"y_{drv}_{k}", cat="Binary") for drv in range(D) for k in range(K)}
    x = {(drv,i,j): pulp.LpVariable(f"x_{drv}_{i}_{j}", cat="Binary")
         for drv in range(D) for (i,j) in arcs_d[drv]}
    u = {(drv,v): pulp.LpVariable(f"u_{drv}_{v}", lowBound=1, upBound=2*K)
         for drv in range(D) for v in real_nodes}
    t = {(drv,v): pulp.LpVariable(f"t_{drv}_{v}", lowBound=0, upBound=M_time)
         for drv in range(D) for v in [dep(drv)]+real_nodes}
    w = {(drv,k): pulp.LpVariable(f"w_{drv}_{k}", lowBound=0, upBound=M_time)
         for drv in range(D) for k in range(K)}

    prob += pulp.lpSum(arc_dist(drv,i,j)*x[drv,i,j] for drv in range(D) for (i,j) in arcs_d[drv])

    for k in range(K):
        prob += pulp.lpSum(y[drv,k] for drv in range(D)) == 1

    for drv in range(D):
        arcs = arcs_d[drv]
        prob += pulp.lpSum(x[drv,i,j] for (i,j) in arcs if i==dep(drv)) == 1
        prob += pulp.lpSum(x[drv,i,j] for (i,j) in arcs if j==snk(drv)) == 1
        for k in range(K):
            r, c = R(k), C(k)
            for nd in [r, c]:
                prob += pulp.lpSum(x[drv,i,j] for (i,j) in arcs if j==nd) == y[drv,k]
                prob += pulp.lpSum(x[drv,i,j] for (i,j) in arcs if i==nd) == y[drv,k]
        for (i,j) in arcs:
            if i in real_nodes and j in real_nodes:
                prob += u[drv,j] >= u[drv,i] + 1 - M_pos*(1-x[drv,i,j])
        for k in range(K):
            prob += u[drv,C(k)] >= u[drv,R(k)] + 1 - M_pos*(1-y[drv,k])
        for (i,j) in arcs:
            if j!=snk(drv) and (drv,j) in t:
                svc = SERVICE if i in cust_set else 0
                prob += t[drv,j] >= t[drv,i] + svc + arc_tt(drv,i,j) - M_time*(1-x[drv,i,j])
        for k in range(K):
            prob += t[drv,R(k)] >= orders[k][2] - M_time*(1-y[drv,k])
        for k in range(K):
            a = orders[k][2]
            prob += w[drv,k] >= t[drv,C(k)] - a - M_time*(1-y[drv,k])
            prob += w[drv,k] <= M_time * y[drv,k]

    prob += pulp.lpSum(w[drv,k] for drv in range(D) for k in range(K)) <= W * K

    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=time_limit))
    status = pulp.LpStatus[prob.status]
    if prob.status != 1: return None, None, None, status

    total_dist = pulp.value(prob.objective)
    waits = []
    for k in range(K):
        for drv in range(D):
            if pulp.value(y[drv,k]) > 0.5:
                waits.append(pulp.value(t[drv,C(k)]) - orders[k][2]); break
    avg_wait, max_wait = sum(waits)/K, max(waits)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Instance: {orders_file} | W={W} | {D} drivers")
        print(f"Status: {status}  |  Total dist: {total_dist:.2f} km  |  Avg wait: {avg_wait:.1f} min  |  Max wait: {max_wait:.1f} min")
        for drv in range(D):
            assigned = [k for k in range(K) if pulp.value(y[drv,k]) > 0.5]
            if not assigned:
                print(f"\n  Driver {drv+1} ({d_locs[drv]}): IDLE"); continue
            arcs = arcs_d[drv]
            route, cur = [dep(drv)], dep(drv)
            while True:
                nxt = next((j for (i,j) in arcs if i==cur and pulp.value(x.get((drv,i,j),0))>0.5), None)
                if nxt is None or nxt==snk(drv): break
                route.append(nxt); cur = nxt
            route_dist = sum(arc_dist(drv,route[i],route[i+1]) for i in range(len(route)-1))
            print(f"\n  Driver {drv+1} ({d_locs[drv]}, {d_speeds[drv]}km/h) | orders {[k+1 for k in assigned]} | dist={route_dist:.2f} km")
            for v in route:
                if v==dep(drv): print(f"    [START   ] {d_locs[drv]}")
                elif v.startswith("R"):
                    k=int(v[1:]); print(f"    [PICKUP  {k+1}] {orders[k][0]}  (arrive={pulp.value(t[drv,v]):.0f}min)")
                elif v.startswith("C"):
                    k=int(v[1:]); wt=pulp.value(t[drv,v])-orders[k][2]
                    print(f"    [DELIVER {k+1}] {orders[k][1]}  (arrive={pulp.value(t[drv,v]):.0f}min, wait={wt:.1f}min)")
    return total_dist, avg_wait, max_wait, status

# ── Run Part III ──────────────────────────────────────────────────────────────
solve_part3("part3_small.csv", "part3_drivers.csv", W=120, verbose=True)



Instance: part3_small.csv | W=120 | 3 drivers
Status: Optimal  |  Total dist: 30.54 km  |  Avg wait: 23.3 min  |  Max wait: 86.0 min

  Driver 1 (Downtown Toronto (Richmond / Adelaide / King), 40km/h) | orders [1, 3, 5] | dist=16.87 km
    [START   ] Downtown Toronto (Richmond / Adelaide / King)
    [PICKUP  5] Downtown Toronto (Kensington Market / Chinatown / Grange Park)  (arrive=1044min)
    [PICKUP  1] Downtown Toronto (Central Bay Street)  (arrive=1045min)
    [DELIVER 5] Downtown Toronto (Central Bay Street)  (arrive=1045min, wait=1.5min)
    [PICKUP  3] Downtown Toronto (Ryerson)  (arrive=1097min)
    [DELIVER 3] York (Fairbank / Oakwood)  (arrive=1107min, wait=10.2min)
    [DELIVER 1] North York (Armour Heights / Wilson Heights / Downsview North)  (arrive=1123min, wait=86.0min)

  Driver 2 (Downtown Toronto (St. James Park), 35km/h) | orders [4] | dist=7.22 km
    [START   ] Downtown Toronto (St. James Park)
    [PICKUP  4] Downtown Toronto (St. James Park)  (arrive=1073min)
 

(30.544085866191324, 23.29333999999999, 85.99309999999991, 'Optimal')

In [8]:
# Q4: Driver sensitivity — how does solution change with fewer drivers?
print(f"{'Drivers':>8}  {'Distance (km)':>14}  {'Avg wait (min)':>15}  {'Max wait':>9}  {'Status':>10}")
print("-"*60)
for nd in range(1, 4):
    d, avg, mx, status = solve_part3("part3_small.csv", "part3_drivers.csv", W=120, n_drivers=nd)
    if d is not None:
        print(f"{nd:>8}  {d:>14.2f}  {avg:>15.1f}  {mx:>9.1f}  {status:>10}")
    else:
        print(f"{nd:>8}  {'—':>14}  {'—':>15}  {'—':>9}  {status:>10}")


 Drivers   Distance (km)   Avg wait (min)   Max wait      Status
------------------------------------------------------------


       1           37.18             53.1      110.8     Optimal


       2           30.18             35.4      100.1     Optimal


       3           30.54             23.3       86.0     Optimal


---
## Part IV — Scaling: Greedy Heuristic

**Insight from Parts I–III:** Drivers close to a restaurant should handle that order.  
Each driver's sub-route can be solved exactly with Part II MIP (fast when orders-per-driver is small).

**Algorithm:**
1. Sort orders by availability time (earliest first).
2. Greedily assign each order to the driver whose current position is closest to the restaurant.
3. Update driver's virtual position to the new customer location.
4. Solve each driver's sub-route independently with the Part II MIP.

**Complexity:** O(K×D) assignment + fast MIP per driver.

In [9]:
def route_single_driver(driver_loc, driver_speed, assigned_orders, W, time_limit=60):
    """Optimally route one driver over their assigned orders (Part II sub-MIP)."""
    if not assigned_orders: return 0.0, []
    K = len(assigned_orders)
    DEPOT = "__DEPOT__"; SINK = "__SINK__"
    def R(k): return f"R{k}"
    def C(k): return f"C{k}"
    rest_nodes = [R(k) for k in range(K)]; cust_nodes = [C(k) for k in range(K)]
    real_nodes = rest_nodes + cust_nodes; cust_set = set(cust_nodes)

    def nloc(v):
        if v==DEPOT: return driver_loc
        k=int(v[1:]); return assigned_orders[k][0] if v.startswith("R") else assigned_orders[k][1]
    def adist(i,j):
        if j==SINK: return 0.0
        return dist_km(dist_map, nloc(i), nloc(j))
    def att(i,j): return adist(i,j)/driver_speed*60.0

    nodes = [DEPOT]+real_nodes+[SINK]
    arcs  = [(i,j) for i in nodes for j in nodes
             if i!=j and j!=DEPOT and i!=SINK and not(j==SINK and i not in cust_set)]

    prob = pulp.LpProblem("Sub", pulp.LpMinimize)
    M_pos=2*K+2; M_t=24*60
    x = {(i,j): pulp.LpVariable(f"x_{i}_{j}", cat="Binary") for (i,j) in arcs}
    u = {v: pulp.LpVariable(f"u_{v}", lowBound=1, upBound=2*K) for v in real_nodes}
    t = {v: pulp.LpVariable(f"t_{v}", lowBound=0, upBound=M_t) for v in [DEPOT]+real_nodes}

    prob += pulp.lpSum((0.0 if j==SINK else adist(i,j))*x[i,j] for (i,j) in arcs)
    prob += pulp.lpSum(x[i,j] for (i,j) in arcs if i==DEPOT) == 1
    prob += pulp.lpSum(x[i,j] for (i,j) in arcs if j==SINK)  == 1
    for v in real_nodes:
        prob += pulp.lpSum(x[i,j] for (i,j) in arcs if j==v) == 1
        prob += pulp.lpSum(x[i,j] for (i,j) in arcs if i==v) == 1
    for (i,j) in arcs:
        if i in real_nodes and j in real_nodes:
            prob += u[j] >= u[i]+1-M_pos*(1-x[i,j])
    for k in range(K): prob += u[C(k)] >= u[R(k)]+1
    for (i,j) in arcs:
        if j!=SINK and j in t:
            svc=SERVICE if i in cust_set else 0
            prob += t[j] >= t[i]+svc+att(i,j)-M_t*(1-x[i,j])
    for k in range(K): prob += t[R(k)] >= assigned_orders[k][2]
    prob += pulp.lpSum(t[C(k)]-assigned_orders[k][2] for k in range(K)) <= W*K

    prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=time_limit))
    if prob.status != 1: return None, None
    return pulp.value(prob.objective), [pulp.value(t[C(k)])-assigned_orders[k][2] for k in range(K)]


def heuristic(orders_file, drivers_file, W=120, verbose=False):
    orders_df  = pd.read_csv(BASE + orders_file)
    drivers_df = pd.read_csv(BASE + drivers_file)
    orders = list(zip(
        orders_df["restaurant"], orders_df["customer"],
        [parse_time(a) for a in orders_df["estimated availability"]]
    ))
    K=len(orders); D=len(drivers_df)
    d_locs=list(drivers_df["start region"]); d_speeds=list(drivers_df["velocity"])

    # Step 1: greedy assignment (sort by availability, pick nearest driver)
    sorted_k = sorted(range(K), key=lambda k: orders[k][2])
    virtual_pos = list(d_locs)
    driver_orders = {drv: [] for drv in range(D)}
    for k in sorted_k:
        r_loc = orders[k][0]
        best = min(range(D), key=lambda d: dist_km(dist_map, virtual_pos[d], r_loc))
        driver_orders[best].append(k)
        virtual_pos[best] = orders[k][1]

    # Step 2: solve each driver's sub-route
    t0=timer.time(); total_dist=0.0; all_waits=[]
    for drv in range(D):
        assigned = [(orders[k][0], orders[k][1], orders[k][2]) for k in driver_orders[drv]]
        d_val, waits = route_single_driver(d_locs[drv], d_speeds[drv], assigned, W)
        if d_val is None: print(f"  Driver {drv+1}: infeasible"); continue
        total_dist += d_val; all_waits.extend(waits)
        if verbose:
            print(f"  Driver {drv+1} ({d_locs[drv]}, {d_speeds[drv]}km/h) | "
                  f"orders {[k+1 for k in driver_orders[drv]]} | dist={d_val:.2f} km")

    avg_wait = sum(all_waits)/len(all_waits) if all_waits else 0
    max_wait = max(all_waits) if all_waits else 0
    print(f"\n  Total distance : {total_dist:.2f} km")
    print(f"  Avg wait       : {avg_wait:.1f} min   Max wait: {max_wait:.1f} min")
    print(f"  Runtime        : {timer.time()-t0:.2f} s")
    return total_dist, avg_wait, max_wait

# ── Evaluate on small instance vs Part III optimal ────────────────────────────
print("Small instance (part3_small.csv) — W=120")
print("Part III optimal: 30.54 km, avg_wait=23.3 min\n")
print("Heuristic:")
heuristic("part3_small.csv", "part3_drivers.csv", W=120, verbose=True)

print("\n" + "="*60)
print("Large instance (part4_large.csv) — W=120\n")
heuristic("part4_large.csv", "part4_drivers.csv", W=120, verbose=True)


Small instance (part3_small.csv) — W=120
Part III optimal: 30.54 km, avg_wait=23.3 min

Heuristic:
  Driver 1 (Downtown Toronto (Richmond / Adelaide / King), 40km/h) | orders [1] | dist=12.65 km
  Driver 2 (Downtown Toronto (St. James Park), 35km/h) | orders [4] | dist=7.22 km
  Driver 3 (Downtown Toronto (Church and Wellesley), 32km/h) | orders [5, 2, 3] | dist=15.88 km

  Total distance : 35.74 km
  Avg wait       : 77.1 min   Max wait: 120.0 min
  Runtime        : 0.12 s

Large instance (part4_large.csv) — W=120

  Driver 1 (Downtown Toronto (Richmond / Adelaide / King), 20km/h) | orders [9] | dist=4.51 km
  Driver 2 (Downtown Toronto (St. James Park), 20km/h) | orders [4] | dist=7.22 km
  Driver 3 (Downtown Toronto (Church and Wellesley), 20km/h) | orders [8] | dist=10.45 km
  Driver 4 (Downtown Toronto (Christie), 20km/h) | orders [2] | dist=3.42 km
  Driver 5 (Downtown Toronto (Church and Wellesley), 20km/h) | orders [10, 6] | dist=9.39 km
  Driver 6 (Downtown Toronto Stn A PO Bo

(99.32821073781868, 86.37060666666666, 237.05050000000006)